# Evaluate your own model with GraphInstruct

This notebook shows how to plug **any LLM** (commercial API or local model)
into the GraphInstruct evaluation pipeline. You'll get back per-level
scores comparable to the 45 baseline cells in [`results/quality/`](../results/quality/).

## What you'll need

- A way to generate text from your model (any callable that takes a string
  and returns a string)
- ~1 minute per (instruction, sample) pair on most APIs
- Optional: GPU for D2 (G-BERTScore) and D3 (Node Classification Gap)
  metrics; CPU works for D1 / D4 / D5


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'examples':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')


## Step 1: Define your model wrapper

GraphInstruct's `run_baseline.py` already supports OpenAI-compatible APIs
(`--provider openai --api-base <URL>`), Anthropic, Alibaba Bailian, and
DeepSeek. If you're plugging in a custom model, write a single function:

```python
def my_model(prompt: str, **kwargs) -> str:
    "Return the LLM's text output."
    ...
```

Below is a minimal stub — replace with your real model.


In [ ]:
def my_model(prompt: str) -> str:
    """Replace this with your real LLM call."""
    # Example: a constant trivial-tree generator (intentionally weak baseline)
    return (
        "Graph[name='trivial', nodes=3] {\n"
        "    node_list = [0, 1, 2];\n"
        "    edge_list = [(0, 1), (1, 2)];\n"
        "}"
    )


## Step 2: Pick a small slice for fast iteration

When wiring up a new model, run on a 10-instruction subset first to
verify everything works before committing to the full 800.


In [ ]:
from graphinstruct.data_loader import load_all_levels

instructions = load_all_levels(data_dir=REPO / 'data' / 'instructions')

# Take 2 from each of L0/L1/L2/L3, 1 from L4/L5 = 10 total
counts = {0: 2, 1: 2, 2: 2, 3: 2, 4: 1, 5: 1}
mini = [
    inst
    for L, n in counts.items()
    for inst in [i for i in instructions[L] if i.feasible][:n]
]
print(f'Mini set: {len(mini)} instructions across L0–L5')


## Step 3: Generate outputs

Loop over the mini set and call your model. With the trivial stub above
this runs in <1 s; with a real API it'll take ~1 minute.


In [ ]:
from tqdm import tqdm

outputs = []
for inst in tqdm(mini, desc='Generating'):
    text = my_model(inst.instruction)
    outputs.append({
        'instruction_id': inst.id,
        'level': inst.level,
        'graph_serialized': text,
    })
print(f'\n{len(outputs)} outputs collected')


## Step 4: Score with D1 + D4 (structural + instruction match)

These are the two highest-weight dimensions across all 6 levels and
require no GPU.


In [ ]:
from collections import defaultdict
import statistics

from graphinstruct.parser import parse
from graphinstruct.metrics.structural import valid_rate
from graphinstruct.metrics.instruction import instruction_score

per_level = defaultdict(list)
for inst, out in zip(mini, outputs):
    try:
        result = parse(out['graph_serialized'])
        g = result.graph
        d1 = valid_rate([g], constraints=list(inst.explicit_constraints))
        d4 = instruction_score(g, list(inst.explicit_constraints))
    except Exception:
        d1, d4 = 0.0, 0.0
    per_level[inst.level].append({'D1': d1, 'D4': d4})

for L in sorted(per_level):
    d1_mean = statistics.mean(r['D1'] for r in per_level[L])
    d4_mean = statistics.mean(r['D4'] for r in per_level[L])
    print(f'L{L}: n={len(per_level[L])}  D1={d1_mean:.3f}  D4={d4_mean:.3f}')


## Step 5: Compare against the 12 baseline LLMs

Once you've evaluated on the full 800 instructions, you can drop your
scores onto the leaderboard for direct comparison.


In [ ]:
import csv
from pathlib import Path

leaderboard = REPO / 'results' / 'leaderboards' / 'tab1_quality_top15.csv'
with open(leaderboard, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = list(reader)
print(f'\n{leaderboard.name}:')
for r in rows[:5]:
    print(f"  #{r['Rank']}  {r['Model']:<25} {r['Strategy']}  Q={r['Quality (Total)']}")


## Going further

- **Full 800-instruction run**: drop the `counts` slice above and loop
  over all 800 instructions; budget ~1 hr for a fast model, ~6 hrs for a
  reasoning model
- **All 5 dimensions**: install `[full]` (`pip install -e ".[full]"`) for
  D2 (BERTScore-based) and D3 (lightweight GCN-based) metrics
- **Use `scripts/run_baseline.py` directly**: handles
  retries, rate limiting, partial-progress checkpointing, and writes the
  same `quality.json` schema used by the 45 baseline cells
